# Setup (only run once)
1. Makes a copy of the required tables to avoid deletion
2. Makes a VS endpoint and index from a delta table

In [0]:
%pip install -qqqq -U -r ../requirements.txt
# Restart to load the packages into the Python environment
dbutils.library.restartPython()

In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
from mlflow.models import ModelConfig

cfg = ModelConfig(development_config="../02_agent/config.yml")
cfg.to_dict()

In [0]:
catalog_name = cfg.get('catalog')
schema_name = cfg.get('schema')

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

## Make a copy of the required tables into your `catalog.schema`
In case they get deleted

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {catalog_name}.{schema_name}.cust_service_data")
spark.sql(f"""
    CREATE TABLE {catalog_name}.{schema_name}.cust_service_data AS 
    SELECT * FROM retail_prod.agents.cust_service_data
""")

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {catalog_name}.{schema_name}.policies")
spark.sql(f"""
    CREATE TABLE {catalog_name}.{schema_name}.policies AS 
    SELECT * FROM retail_prod.agents.policies
""")

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {catalog_name}.{schema_name}.product_docs")
spark.sql(f"""
    CREATE TABLE {catalog_name}.{schema_name}.product_docs AS 
    SELECT * FROM retail_prod.agents.product_docs
""")

In [0]:
%sql
SHOW TBLPROPERTIES yen_training.agents.product_docs

In [0]:
spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.product_docs SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

## Create the Vector Store endpoint and index from the source delta table
See more in this [VS guide](https://docs.databricks.com/aws/en/generative-ai/create-query-vector-search)

In [0]:
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()

# Create VS endpoint
# Check if the endpoint already exists
endpoints = client.list_endpoints()
endpoint_name = cfg.get('retriever')['vs_endpoint']
if endpoint_name not in [ep.get('name') for ep in endpoints['endpoints']]:
    # Create VS endpoint
    client.create_endpoint(
        name=endpoint_name,
        endpoint_type="STANDARD"
    )

# Create VS index
index = client.create_delta_sync_index(
  endpoint_name=cfg.get('retriever')['vs_endpoint'],
  source_table_name=cfg.get('retriever')['vs_source'],
  index_name=cfg.get('retriever')['vs_index'],
  pipeline_type="TRIGGERED",
  primary_key="product_id",
  embedding_source_column="product_doc",
  embedding_model_endpoint_name="databricks-gte-large-en"
)

## Set up Lakebase
You only need this if you need a Postgres database backend (usually in production) for your agent memory
https://www.databricks.com/blog/how-use-lakebase-transactional-data-layer-databricks-apps

In [0]:
from databricks.sdk import WorkspaceClient

instance_name = cfg.get("lakebase").get("instance_name")
w = WorkspaceClient(
    host=cfg.get("host"),
    client_id=cfg.get("lakebase").get("client_id"),
    client_secret=cfg.get("lakebase").get("client_secret")
)

In [0]:
from databricks.sdk.service.database import DatabaseInstance

instance = w.database.create_database_instance(
   DatabaseInstance(
       name=instance_name,
       capacity="CU_1"
   )
)

#### Try connecting to Lakebase
Check out [various ways](https://docs.databricks.com/aws/en/oltp/query#ways-to-access-your-database) of connecting

In [0]:
import sys
sys.path.append('/Workspace/Users/yen.low@databricks.com/databricks_materials/agents-on-databricks/')
sys.path

In [0]:
from langgraph.checkpoint.postgres import PostgresSaver
from helper import LakebaseConnect
from databricks.sdk import WorkspaceClient

dbClient = LakebaseConnect(
    user = cfg.get("lakebase").get("client_id"),
    password = None, # leave None to generate ephemeral token (1h)
    instance_name = cfg.get("lakebase").get("instance_name"), 
    database = cfg.get("lakebase").get("database"),
    wsClient = w
)
dbClient.test_query()

### Add your service principal to Lakebase
1. Add the SP as a new role in Lakebase
```
Hamburger menu > Compute > Lakebase > <your_instance> > Permissions > Add PostgresSQL role
```
Paste the `DATABRICKS_CLIENT_ID` into the Workspace identity field and click confirm
![](./lakebase_addSP.png)

2. Grant the necessary permissions
```
Hamburger menu > Compute > Lakebase > <your_instance> > New Query
```
<br><br>
[Set a Postgres password for the SP](https://docs.databricks.com/aws/en/oltp/authentication?language=Python+SDK#authenticate-with-postgres-roles-and-passwords)
```
CREATE ROLE "2e962c53-81fb-407e-afbb-2a60e6c69db6" LOGIN PASSWORD "ud664DeJyP";
```
<br><br>
```
SELECT table_name FROM information_schema.tables WHERE table_schema='public';
GRANT ALL PRIVILEGES ON SCHEMA public TO "<DATABRICKS_CLIENT_ID>";
-- The checkpoints table will be created when you first run langgraph checkpointer setup
GRANT SELECT, INSERT, UPDATE, DELETE ON TABLE <your_database>.public.checkpoints TO "<DATABRICKS_CLIENT_ID>";
GRANT SELECT, INSERT, UPDATE, DELETE ON TABLE <your_database>.public.checkpoints_blobs TO "<DATABRICKS_CLIENT_ID>";
GRANT SELECT, INSERT, UPDATE, DELETE ON TABLE <your_database>.public.checkpoint_writes TO "<DATABRICKS_CLIENT_ID>";
GRANT SELECT, INSERT, UPDATE, DELETE ON TABLE <your_database>.public.checkpoint_migrations TO "<DATABRICKS_CLIENT_ID>";
```
<br>
Check privileges
<br>

```
SELECT * FROM information_schema.role_table_grants 
WHERE grantee = '<DATABRICKS_CLIENT_ID>';
```
